# 摄像预览交付版 1.1.3
日常使用：检查 → 启动 → 使用网页 → 正常停止。逐格执行，不要 Run All。无需性能对照。


## 1 工程检查
正常停止旧应用，打开本目录新内核。


In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT=Path.cwd().resolve()
if PROJECT_ROOT.name=='notebooks': PROJECT_ROOT=PROJECT_ROOT.parent
if not (PROJECT_ROOT/'project.json').is_file(): raise RuntimeError('从本工程notebooks目录打开本Notebook')
if 'psv7_pynq' in sys.modules and Path(sys.modules['psv7_pynq'].__file__).resolve().parent!=PROJECT_ROOT:
    raise RuntimeError('内核加载了另一工程；先正常停止旧应用，再使用本Notebook的新内核')
if 'session' in globals() and any(getattr(session,k,None) is not None for k in ('app','audio','host_service','_process_lock')):
    raise RuntimeError('已有会话尚未完整停止，请保留当前内核和Owner')
sys.path.insert(0,str(PROJECT_ROOT.parent))
from psv7_pynq.app.session import PynqSession
session=PynqSession(PROJECT_ROOT)
session.check()


## 2 启动
旧应用正常停止且没有断电，保留reuse/False；实际断电重新上电才改cold_start/True。BOARD_IP填写当前地址。


In [ ]:
STARTUP_MODE='reuse'  # 旧应用正常停止且未断电：reuse；真正断电上电后：cold_start
CLEAN_BOARD_POWER_UP=False  # 真正断电重新上电才用True，并配合cold_start
BOARD_IP='192.168.2.99'  # 改用其它网口IP时同步修改
app=session.start(mode=STARTUP_MODE,clean_boot=CLEAN_BOARD_POWER_UP)
try:
    host_info=session.start_host(host='0.0.0.0',port=8765,allowed_hosts=[BOARD_IP])
except Exception:
    session.stop()
    raise
{'version':session.config['project_version'],
 'camera_handoff_policy':app.camera_handoff_policy,
 'host':host_info}


## 3 正常停止并保存
完成使用后执行。无需 compare_result。成功后打印停止报告并提供下载链接；重复执行复用本会话报告。停止失败保留内核及 owner，不强制释放。


In [ ]:
# Daily stop: independent of comparison or recognition result variables.
from pathlib import Path
import json, time, zipfile
from IPython.display import display, FileLink

if 'session' not in globals():
    raise RuntimeError('本内核没有 session，请回到实际运行应用的 Notebook 正常停止')
if session.app is not None:
    _stop_folder = Path(session.app.report_folder)
    stop_report = session.stop()
    # Cache immediately: an output/write failure must not trigger another stop.
    session._preview_delivery_stop = (stop_report, _stop_folder)
elif hasattr(session, '_preview_delivery_stop'):
    stop_report, _stop_folder = session._preview_delivery_stop
else:
    raise RuntimeError('本会话没有活动应用或已保存的停止报告；无需重新启动来停止')

_stop_folder.mkdir(parents=True, exist_ok=True)
_stop_file = _stop_folder / 'normal_stop.json'
_stop_file.write_text(json.dumps(stop_report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(json.dumps(stop_report, ensure_ascii=False, indent=2))
_memory = stop_report.get('memory', {})
_host = stop_report.get('host', {})
_model = stop_report.get('model3d', {})
_driver = stop_report.get('driver_after', {})
_clean = (
    stop_report.get('errors') == []
    and stop_report.get('retained_buffers') == 0
    and stop_report.get('dma_owned') is False
    and stop_report.get('owned_gpio') == []
    and stop_report.get('preview_thread_alive') is False
    and stop_report.get('voice_worker_alive') is False
    and stop_report.get('request_controller_closed') is True
    and _memory.get('active_bos') == 0 and _memory.get('active_bytes') == 0
    and _memory.get('allocated_bos') is not None
    and _memory.get('allocated_bos') == _memory.get('released_bos')
    and _driver.get('num_buffers') == 0 and _driver.get('mem_used') == 0
    and _host.get('host_running') is False
    and not _host.get('http_thread_alive', False)
    and not _host.get('command_thread_alive', False)
    and _model.get('closed') is True and _model.get('active') is False
)
if not _clean:
    raise RuntimeError('停止报告未满足完整释放条件，报告已保存；保留内核和 owner，不强制释放')
_stop_zip = Path.cwd() / ('preview_delivery_stop_' + str(time.time_ns()) + '.zip')
with zipfile.ZipFile(_stop_zip, 'x', zipfile.ZIP_DEFLATED) as _zip:
    _zip.write(_stop_file, 'normal_stop.json')
print('正常停止通过；无需性能对照。停止报告已保存：', _stop_file)
display(FileLink(_stop_zip.name))
